# Executable Sakurai: Dynamics of Coherent States
**Bridging rigorous quantum formalism with interactive Python simulations.**

Welcome to *Executable Sakurai*. In this notebook, we explore the properties and time-evolution of the **Coherent State** $|\alpha\rangle$ under the Quantum Harmonic Oscillator (QHO) Hamiltonian.

In standard quantum mechanics, wavepackets typically disperse over time due to quantum interference. However, the coherent state—often dubbed the "most classical" of all quantum states—maintains its minimum-uncertainty wavepacket shape, oscillating smoothly within a quadratic potential perfectly in sync with classical equations of motion. 

Below, we digitize my handwritten derivations based on J.J. Sakurai's *Modern Quantum Mechanics*, followed by an interactive numerical simulation using `QuTiP`.

## 1. The Raw Derivation
Before translating this into executable code, it is crucial to understand the foundational operator algebra. Below is my original handwritten manuscript deriving the Fock space expansion, uncertainty principles, and time-evolution of the coherent state.
<img src="Handwritten img/Coherent_1.png" width="800">


## 2. Mathematical Formalism: Expansion in Fock Basis

We define the coherent state $|\alpha\rangle$ as the right eigenket of the non-Hermitian annihilation operator $\hat{a}$:
$$\hat{a}|\alpha\rangle = \alpha|\alpha\rangle$$
where $\alpha$ is a complex number. 

To represent this state computationally, we must expand it in the basis of energy eigenstates (Fock states) $|n\rangle$. Assuming an expansion $|\alpha\rangle = \sum c_n |n\rangle$, we apply the annihilation operator:
$$\sum c_n \sqrt{n} |n-1\rangle = \alpha \sum c_n |n\rangle$$

By matching the coefficients, we obtain the recursive relation $c_n = c_0 \frac{\alpha^n}{\sqrt{n!}}$. Normalizing the state ($\langle\alpha|\alpha\rangle = 1$) yields $c_0 = \exp(-|\alpha|^2/2)$. Thus, the rigorous expansion is:

$$|\alpha\rangle = e^{-\frac{|\alpha|^2}{2}} \sum_{n=0}^{\infty} \frac{\alpha^n}{\sqrt{n!}} |n\rangle$$

> **Computational Insight:** In Python (`QuTiP`), a coherent state is generated by truncating this infinite sum to a finite Hilbert space dimension $N$. The expected photon number is $\langle \alpha | \hat{N} | \alpha \rangle = |\alpha|^2$. We must choose $N \gg |\alpha|^2$ to avoid truncation errors in our simulation.

In [ ]:
import numpy as np
from qutip import destroy, coherent

# 1. System Parameters
N = 20           # Truncated Hilbert space dimension (Fock states 0 to 19)
alpha = 2.0 + 0j # Complex eigenvalue for the coherent state

# 2. Operators and State Vectors
a = destroy(N)          
psi = coherent(N, alpha) 

# 3. The Operator Shift vs. Scalar Multiplication
a_psi = a * psi 
alpha_psi = alpha * psi

print("--- Verifying the Eigenvalue Equation: a|α> = α|α> ---")
print(f"First 5 amplitudes of |α>:            {np.round(psi.full()[:5].flatten(), 4)}")
print(f"First 5 amplitudes of a|α> (shifted): {np.round(a_psi.full()[:5].flatten(), 4)}")
print(f"First 5 amplitudes of α * |α>:        {np.round(alpha_psi.full()[:5].flatten(), 4)}\n")

print("--- Computational Verification ---")
is_exact_match = np.allclose(a_psi.full(), alpha_psi.full())
print(f"Do the vectors match exactly across all N={N} dimensions? {is_exact_match}")

### The Truncation Trap: Why did it return `False`?

On a theoretical chalkboard, the Hilbert space is infinite-dimensional, ensuring the fundamental commutation relation $[\hat{a}, \hat{a}^\dagger] = 1$ holds perfectly. But why does a strict numerical verification in our code return `False`?

This is not a bug in our code; rather, we have run into a classic pitfall in computational quantum physics: the **Truncation Trap**.

1. **Boundary Breakdown:** In computational simulations, we must truncate the infinite Hilbert space to a finite dimension (here, $N=20$). When the annihilation operator $\hat{a}$ acts on the highest boundary state $|N-1\rangle$, it attempts to pull amplitude from a non-existent $|N\rangle$ state. This artificial cutoff explicitly breaks the commutation relation at the boundary.
2. **Black-box Algorithmic Error:** By default, `QuTiP` does *not* directly plug into our derived $c_n$ coefficient formula! Instead, it generates the coherent state using the matrix exponential of the **Displacement Operator**: $|\alpha\rangle = \exp(\alpha \hat{a}^\dagger - \alpha^* \hat{a})|0\rangle$. Because the commutation relation is violated in finite-dimensional matrices, the continuous-space Baker-Campbell-Hausdorff (BCH) expansion is no longer strictly valid. This introduces a minuscule, yet mathematically fatal, truncation error.

To fix this, we must "teach" the computer to think like a theoretical physicist: we will abandon the default matrix exponential, force the use of our hand-derived **Analytical Formula**, and safely evaluate the state within the physical subspace (ignoring the artificial boundary).

In [ ]:
print("--- Resolving the Truncation Trap ---")

psi_analytic = coherent(N, alpha, method='analytic')

# 1. Apply operators again
a_psi_ana = a * psi_analytic
alpha_psi_ana = alpha * psi_analytic

# 2. Verify in the "Safe Physical Subspace"
# We slice the arrays [:-2] to completely ignore the top 2 artificial boundary states 
safe_match_analytic = np.allclose(a_psi_ana.full()[:-2], alpha_psi_ana.full()[:-2])

print(f"Do they match in the safe physical subspace using the analytical method? {safe_match_analytic}")
print("\nTheoretical victory! The mathematics holds perfectly when computational boundaries are respected.")

## 3. The Minimum Uncertainty State

Why is the coherent state considered "classical"? We look at the variances in position $\hat{x}$ and momentum $\hat{p}$.
Using the creation and annihilation operators:
$$\hat{x} = \sqrt{\frac{\hbar}{2m\omega}} (\hat{a}^\dagger + \hat{a}), \quad \hat{p} = i\sqrt{\frac{m\hbar\omega}{2}} (\hat{a}^\dagger - \hat{a})$$

Evaluating the expectation values $\langle \hat{x}^2 \rangle - \langle \hat{x} \rangle^2$, we find:
$$\Delta x = \sqrt{\frac{\hbar}{2m\omega}}, \quad \Delta p = \sqrt{\frac{m\hbar\omega}{2}}$$

Hence, the uncertainty product hits the absolute theoretical minimum allowed by Heisenberg's principle:
$$\Delta x \cdot \Delta p = \frac{\hbar}{2}$$

## 4. Time Evolution: The Rigid Wavepacket

The most fascinating property of the coherent state emerges when we evolve it in time under the Harmonic Oscillator Hamiltonian $H = \hbar\omega(\hat{a}^\dagger \hat{a} + \frac{1}{2})$. 

Applying the time-evolution operator $\hat{U}(t) = \exp(-i\hat{H}t/\hbar)$ to our Fock expansion:

$$|\alpha, t\rangle = e^{-i\hat{H}t/\hbar} |\alpha\rangle = e^{-\frac{|\alpha|^2}{2}} \sum_{n=0}^{\infty} \frac{\alpha^n}{\sqrt{n!}} e^{-i\omega(\frac{1}{2} + n)t} |n\rangle$$

By factoring out the zero-point energy phase $e^{-i\omega t/2}$, we can absorb the dynamic phase into the eigenvalue $\alpha$:

$$|\alpha, t\rangle = e^{-i\omega t / 2} \left[ e^{-\frac{|\alpha|^2}{2}} \sum_{n=0}^{\infty} \frac{(\alpha e^{-i\omega t})^n}{\sqrt{n!}} |n\rangle \right]$$

$$|\alpha, t\rangle = e^{-i\omega t / 2} \left| \alpha e^{-i\omega t} \right\rangle$$

> **Physical Insight (The Core of the Simulation):** > The state *remains a coherent state* at all times! The complex eigenvalue $\alpha$ simply rotates in the complex plane: $\alpha(t) = \alpha(0) e^{-i\omega t}$. 
> In phase space (the Wigner function), this corresponds to a rigid Gaussian wavepacket moving in a perfect circle, exactly mimicking a classical pendulum, without spreading!


## 5. Interactive Simulation: QuTiP Implementation
We have demonstrated the static properties of coherent states. Now, let’s explore their dynamic behavior using dynamic phase-space visualization (Wigner Function).

This interactive simulation demonstrates the most profound property of **Coherent States $|\alpha\rangle$**: they are **Rigid Wavepackets**. While any standard superposition of energy eigenstates disperses over time, losing its initial shape and orthogonality, a coherent state rigidly rotates in phase space without changing its Gaussian geometry, preserving its quantum fidelity with the initial state perfectly at full periods.

Run the cell below and interact with the sliders:
* **Time Slider (t/π):** Watch how the states evolve. 


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from qutip import destroy, coherent, fock, variance, wigner, squeeze
import ipywidgets as widgets
from ipywidgets import interact

plt.style.use('seaborn-v0_8-whitegrid')

def interactive_dynamics_showdown_v2(time_frac_pi):
    N = 25 
    a = destroy(N)
    H = a.dag() * a # Hamiltonian (hbar=w=1)
    
    # Coherent State (Rigid)
    alpha0 = 2.0
    psi_coh_0 = coherent(N, alpha0, method='analytic')
    
    # Squeezed State (As a non-rigid comparison)
    z = 0.5
    psi_sq_0 = squeeze(N, z) * fock(N, 0)
    
    # define quadrature operators for uncertainty calculation
    x_op = (a.dag() + a) / np.sqrt(2)
    p_op = 1j * (a.dag() - a) / np.sqrt(2)

    time = time_frac_pi * np.pi
    U = (-1j * H * time).expm()
    
    psi_coh_t = U * psi_coh_0
    psi_sq_t = U * psi_sq_0

    vec = np.linspace(-5, 5, 80)
    W_coh_t = wigner(psi_coh_t, vec, vec, g=np.sqrt(2))
    W_sq_t = wigner(psi_sq_t, vec, vec, g=np.sqrt(2))
    
    uncert_coh_t = np.sqrt(variance(x_op, psi_coh_t) * variance(p_op, psi_coh_t))
    uncert_sq_t = np.sqrt(variance(x_op, psi_sq_t) * variance(p_op, psi_sq_t))
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    w_lim = 1/np.pi
    axes_extent = [vec.min(), vec.max(), vec.min(), vec.max()]
    norm = colors.Normalize(-w_lim, w_lim)
    
    # Panel 1: Squeezed State
    im1 = axes[0].imshow(W_sq_t, cmap='RdBu', norm=norm,
                        extent=axes_extent, origin='lower', aspect='equal',
                        interpolation='gaussian')
    axes[0].set_title("Squeezed State $|\\alpha, z\\rangle$\n" + f"$\\Delta x \\Delta p$ = {uncert_sq_t:.4f}", fontsize=14)
    axes[0].set_xlabel("Position x", fontsize=12)
    axes[0].set_ylabel("Momentum p", fontsize=12)
    
    # Panel 2: Coherent State
    im2 = axes[1].imshow(W_coh_t, cmap='RdBu', norm=norm,
                        extent=axes_extent, origin='lower', aspect='equal',
                        interpolation='gaussian')
    axes[1].set_title("Coherent State $|\\alpha(t)\\rangle$\n" + f"$\\Delta x \\Delta p$ = {uncert_coh_t:.4f}", fontsize=14)
    axes[1].set_xlabel("Position x", fontsize=12)
    

    for ax in axes:
        ax.grid(False)
        ax.axhline(0, color='black', linewidth=0.5, alpha=0.5)
        ax.axvline(0, color='black', linewidth=0.5, alpha=0.5)


    fig.subplots_adjust(right=0.88, top=0.85)
    cbar_ax = fig.add_axes([0.91, 0.15, 0.02, 0.7])
    fig.colorbar(im1, cax=cbar_ax, label='Wigner Function $W(x,p)$')
    
    plt.show()
  
# Interactive widget to explore the dynamics
interact(interactive_dynamics_showdown_v2, 
         time_frac_pi=widgets.FloatSlider(min=0.0, max=2.0, step=0.1, value=0.0,
                                         description='Time :', 
                                         continuous_update=False));

### Physics Analysis: Rigid Rotation vs. Quantum Dispersion

The simulation above provides a high-fidelity visualization of **Ehrenfest's Theorem** and the uniqueness of Coherent States.

#### 1. The Rigid Wavepacket (The Coherent Advantage)
Observe the **Coherent State** (Right Panel). As time $t$ evolves, the wavepacket undergoes a **rigid rotation** in phase space.
* **Shape Invariance:** Its Wigner distribution remains a perfect circle. It does not spread, disperse, or deform.
* **Overlap Fidelity:** Because the shape is invariant, the overlap with the initial state $|\langle \psi(0) | \psi(t) \rangle|$ follows a simple periodic revival. It is the "closest" a quantum state can get to a classical point particle.
* **Minimum Uncertainty:** It remains locked at $\Delta x \Delta p = 0.5$, the absolute floor of Heisenberg's limit.

#### 2. Non-Rigid Dynamics (Ordinary/Squeezed States)
In contrast, look at the **Squeezed State** (Left Panel) representing "ordinary" non-coherent wavepackets.
* **The "Breathing" Effect:** While the total area in phase space is conserved (unitary evolution), the wavepacket "breathes." The uncertainty oscillates between Position ($x$) and Momentum ($p$). 
* **Rapid Dephasing:** For more complex arbitrary states (like a high-level Fock state or a Schrödinger Cat state), the wavepacket would quickly exhibit complex interference patterns.
* **Overlap Decay:** In a real-world potential (even slightly anharmonic), these non-coherent states would disperse rapidly. Their overlap with the initial state $|\langle \psi(0) | \psi(t) \rangle|$ would trend toward **zero** much faster than a coherent state, which retains its localized "packet" identity.
